# EnsemblePredictor Tutorial

This notebook demonstrates how to load and use the `EnsemblePredictor` class
for making PROTAC/degrader activity predictions with uncertainty quantification.

## Overview

The `EnsemblePredictor` combines predictions from multiple models (XGBoost and
MLP) trained across different cross-validation folds and feature sets. It
provides:

- **Weighted ensemble predictions** from all loaded models
- **Multi-task support** — models are grouped by their training task (e.g., Dmax, DC50, Binary)
- **Uncertainty estimates** (standard deviation, confidence intervals)
- **Two types of 95% CI**: percentile-based (non-parametric) and SEM-based (parametric)
- **Individual model predictions** for further analysis
- **Automatic feature encoding** via the saved datamodules
- **Categorical dropdown values** extracted from training encoders

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

from tackai.ensemble_predictor import EnsemblePredictor, SampleInput

# Disable warnings for cleaner output
warnings.filterwarnings("ignore")

## 1. Loading the Ensemble

### 1.1 Loading from a directory (uniform weights)

The simplest way to create an `EnsemblePredictor` is to point it at a
directory that contains model checkpoints (`.ckpt` for MLP, `.json` for
XGBoost) and their associated datamodule files (`_hparams.yaml` +
`_state.pt`).  All models receive equal weight.

> **Note:** Every model must have a corresponding datamodule. If any model
> is missing its datamodule, loading will fail with an error.

After downloading and unzipping files from [this download link](https://zenodo.org/records/15691822/files/ensembles.zip?download=1), we will assume a `ensembles/` directory with subfolders for the different ensemble types (e.g., `bin_caruana_ensemble/`).

> **Note:** Each ensemble loads N models and N Lightning data modules, so it can
> require significant memory to run. We advice to not load too many ensembles.

In [ ]:
PROJ_DIR = Path("/cephyr/users/ribes/Alvis/mimer/stefano/TACK")
ENSEMBLE_CHECKPOINT_DIR = PROJ_DIR / Path("ensembles/ensembles/bin_best_arch_ensemble/")
ENSEMBLE_WEIGHTS_PATH = PROJ_DIR / Path("ensembles/ensembles/bin_best_arch_ensemble/ensemble_weights_bin_caruana_all_models.json")

predictor = EnsemblePredictor.from_directory(
    model_dir=ENSEMBLE_CHECKPOINT_DIR,
    device="cpu",
)
print(predictor)

## 2. Making Predictions

### 2.1 Single prediction with `SampleInput`

The `SampleInput` dataclass wraps all the possible input fields.
**SMILES**, **POI** (name or sequence), and **E3 Ligase** are required.
Optional fields (cell line, assay, treatment time) are filled with defaults
when omitted.

> **Important:** `predict()` now returns a `Dict[str, EnsemblePrediction]`
> mapping each task (e.g. `'dmax'`) to its ensemble result. When only one
> task is present the dict will have a single entry.

### 1.3 Inspecting the loaded models

You can inspect which models were loaded, their types, and weights.

In [ ]:
info = predictor.get_model_info()

print(f"Task(s):         {info['available_tasks']}")
print(f"Models types:    {list(set(list(info['model_types'].values())))}")
print(f"Total models:    {info['n_models']}")
print()

# Count by type
type_counts = Counter(info["model_types"].values())
for model_type, count in type_counts.items():
    print(f"  {model_type}: {count}")

In [ ]:
for k, v in predictor.datamodules.items():
    print(f"{k}: {v}")

for k, v in predictor.model_types.items():
    print(f"{k}: {v}")

In [ ]:
# Show individual model weights (sorted by weight, descending)
weights_df = pd.DataFrame([
    {"model": name, "type": info["model_types"][name], "weight": w}
    for name, w in info["weights"].items()
]).sort_values("weight", ascending=False).reset_index(drop=True)

weights_df

## 2. Making Predictions

### 2.1 Single prediction with `SampleInput`

The `SampleInput` dataclass wraps all the possible input fields.  Only
`smiles` is strictly required — missing fields are filled with sensible
defaults.

In [ ]:
# All models in this ensemble require POI_Sequence (not just POI_Name)
AR_SEQ = "MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAASAAPPGASLLLLQQQQQQQQQQQQQQQQQQQQQQQETSPRQQQQQQGEDGSPQAHRRGPTGYLVLDEEQQPSQPQSALECHPERGCVPEPGAAVAASKGLPQQLPAPPDEDDSAAPSTLSLLGPTFPGLSSCSADLKDILSEASTMQLLQQQQQEAVSEGSSSGRAREASGAPTSSKDNYLGGTSTISDNAKELCKAVSVSMGLGVEALEHLSPGEQLRGDCMYAPLLGVPPAVRPTPCAPLAECKGSLLDDSAGKSTEDTAEYSPFKGGYTKGLEGESLGCSGSAAAGSSGTLELPSTLSLYKSGALDEAAAYQSRDYYNFPLALAGPPPPPPPPHPHARIKLENPLDYGSAWAAAAAQCRYGDLASLHGAGAAGPGSGSPSAAASSSWHTLFTAEEGQLYGPCGGGGGGGGGGGGGGGGGGGGGGGEAGAVAPYGYTRPPQGLAGQESDFTAPDVWYPGGMVSRVPYPSPTCVKSEMGPWMDSYSGPYGDMRLETARDHVLPIDYYFPPQKTCLICGDEASGCHYGALTCGSCKVFFKRAAEGKQKYLCASRNDCTIDKFRRKNCPSCRLRKCYEAGMTLGARKLKKLGNLKLQEEGEASSTTSPTEETTQKLTVSHIEGYECQPIFLNVLEAIEPGVVCAGHDNNQPDSFAALLSSLNELGERQLVHVVKWAKALPGFRNLHVDDQMAVIQYSWMGLMVFAMGWRSFTNVNSRMLYFAPDLVFNEYRMHKSRMYSQCVRMRHLSQEFGWLQITPQEFLCMKALLLFSIIPVDGLKNQKFFDELRMNYIKELDRIIACKRKNPTSCSRRFYQLTKLLDSVQPIARELHQFTFDLLIKSHMVSVDFPEMMAEIISVQVPKILSGKVKPIYFHTQ"
E3_SEQ = "MAGEGDQQDAAHNMGNHLPLLPAESEEEDEMEVEDQDSKEAKKPNIINFDTSLPTSHTYLGADMEEFHGRTLHDDDSCQVIPVLPQVMMILIPGQTLPLQLFHPQEVSMVRNLIQKDRTFAVLAYSNVQEREAQFGTTAEIYAYREEQDFGIEIVKVKAIGRQRFKVLELRTQSDGIQQAKVQILPECVLPSTMSAVQLESLNKCQIFPSKPVSREDQCSYKWWQKYQKRKFHCANLTSWPRWLYSLYDAETLMDRIKKQLREWDENLKDDSLPSNPIDFSYRVAACLPIDDVLRIQLLKIGSAIQRLRCELDIMNKCTSLCCKQCQETEITTKNEIFSLSLCGPMAAYVNPHGYVHETLTVYKACNLNLIGRPSTEHSWFPGYAWTVAQCKICASHIGWKFTATKKDMSPQKFWGLTRSALLPTIPDTEDEISPDKVILCL"

sample = SampleInput(
    smiles="CC1(C)[C@H](NC(=O)c2ccc(N3CCN(CCCOc4ccc(C(=O)NC5CCC(=O)NC5=O)nc4)CC3)nc2)C(C)(C)[C@H]1Oc1ccc(C#N)c(Cl)c1",
    poi_name="AR",
    poi_sequence=AR_SEQ,
    ligase_name="CRBN",
    ligase_sequence=E3_SEQ,
    cell_line="Unknown",
    assay_type="Unknown",
    treatment_time=24.0,
)

# predict() returns a dict: task_name → EnsemblePrediction
# EXAMPLE: Use task_results['dmax'] for Dmax predictions
task_results = predictor.predict(sample, tasks=['bin'])

# For a single-task ensemble, extract the first (only) result
task_name = list(task_results.keys())[0]
result = task_results[task_name]
print(f"Task: {task_name}")
print(result.summary())
print('')
print("Full result as dict:")
print(result.to_dict())

In [ ]:
smiles_list = [
    'Cn1c(=O)n(C2CCC(=O)NC2=O)c2cccc(C#CCCN3CCC4(CC3)CC(n3cc(NC(=O)c5cnn6ccc(N7C[C@H]8C[C@@H]7CO8)nc56)c(C(F)F)n3)C4)c21',
    'Cc1ncsc1C1=CCC([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)C(c2cc(N3CCC(CN4CCC(n5nc(NC(C)C)c6nnc(-c7cccc(F)c7O)cc65)CC4)CC3)no2)C(C)C)C=C1',
    'COc1ccccc1C(=O)NCc1ccc(-c2nn3c(c2C(N)=O)Nc2ccc(N4CCN(CC5CCN(c6ccc(N7CCC(=O)NC7=O)cc6)CC5)CC4)cc2CC3)c(OC)c1',
    'C[C@H]1CC2(CCN(c3ccc(C(=O)N4CCC(N5CCC(N6CCN(c7ccc(N8CCC(=O)NC8=O)cc7)CC6)CC5)CC4)cc3)CC2)CN1c1ccc(C#N)c(Cl)c1',
    'COc1cc(N2CCC(N3CCN(C(=O)C4CCN(c5ccc6c(c5)C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)CC3)CC2)c(C)cc1Nc1ncc(Cl)c(Nc2cccc3c2N(S(C)(=O)=O)CC3)n1',
    'CC(C1CN(c2cc3c(cc2F)C(=O)N(C2CCC(=O)NC2=O)C3=O)C1)N1CCC(c2cn3cc(NC(=O)c4cccc(C5CC5(F)F)n4)c(C(C)(C)O)c(F)c3n2)CC1',
    'O=C1CCC(N2C(=O)c3cccc(NC(=O)C[C@@H]4CCN(CCc5cc(O)c(N6CC(=O)NS6(=O)=O)c(F)c5)C4)c3C2=O)C(=O)N1',
    'Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)CCCCCCCCCCCN(C)CCOC23CC4(C)CC(C)(CC(Cn5ncc(-c6ccc(N7CCCc8c7nnc(Nc7nc9ccccc9s7)c8C)nc6C(=O)O)c5C)(C4)C2)C3)C(C)(C)C)cc1',
    'Cc1cccc(NC(=O)c2ccc(-c3nn4c(c3C(N)=O)Nc3ccc(N5CCN(CC6CCN(c7ccc(N8CCC(=O)N(CC(=O)OC(C)(C)C)C8=O)cc7)CC6)CC5)cc3CC4)c(F)c2C)n1',
    'Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)CCCCCCC(=O)N2CCN(CC[C@H](CSc3ccccc3)Nc3ccc(S(=O)(=O)NC(=O)c4ccc5c(c4)OC[C@@H]4CN(CC6=C(c7ccc(Cl)cc7)CCC(C)(C)C6)CCN54)cc3S(=O)(=O)C(F)(F)F)CC2)C(C)(C)C)cc1',
    'COc1cc2nn(C3CC4(CCC(C(=O)N5CCN(c6ccc7c(c6)C(=O)N(C6CCC(=O)NC6=O)C7=O)CC5)CC4)C3)cc2cc1NC(=O)c1cccc(C(F)(F)F)n1',
    'Nc1ncnc2c1c(-c1ccc(Oc3ccccc3)cc1)nn2C1CCN(CCCCCCOCCOCC(=O)Nc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)CC1',
    'Cn1c(=O)n(C2CCC(=O)NC2=O)c2cccc(C#CCN3CC[C@@H](CN4CCC5(CC4)CC(n4cc(NC(=O)c6cnn7ccc(N8C[C@H]9C[C@@H]8CO9)nc67)c(C(F)F)n4)C5)C3)c21',
    'CC1(C)CC(=C(c2ccc(N3CCC(CN4CCN(c5ccc(C(=O)N[C@H]6CCC(=O)NC6=O)c(F)c5)CC4)CC3)nc2)c2ccc3[nH]nc(F)c3c2)CC(C)(C)O1',
    'O=C1CCC(n2ncc(N3CCN(CC4CCN(c5ccc([C@@H]6c7ccc(O)cc7CC[C@@H]6c6ccccc6)cc5)CC4)CC3)cc2=O)C(=O)N1',
    'Nc1ncnc2c1c(-c1ccc(Oc3ccccc3)cc1)nn2[C@H]1CC[C@@H](N2CCN(C3CN(c4ccc5c(c4)C(=O)N([C@@H]4CCC(=O)NC4=O)C5=O)C3)CC2)CC1',
    'CCn1c(C)c(S(C)(=O)=O)c(CCc2cc(F)cc(NS(=O)(=O)c3ccc(NC(CCN4CCN(C(=O)CCCOCC(=O)N[C@H](C(=O)N5C[C@H](O)C[C@H]5C(=O)N[C@@H](C)c5ccc(-c6scnc6C)cc5)C(C)(C)C)CC4)CSc4ccccc4)c([N+](=O)[O-])c3)c2)c1-c1ccc(F)cc1',
    'CCCNc1nn(C2CCN(CC3CCN(CC(=O)N[C@H](C(=O)N4C[C@H](O)C[C@H]4C(=O)N[C@@H](C)c4ccc(-c5scnc5C)cc4)C(C)(C)C)CC3)CC2)c2cc(-c3ccccc3O)nnc12',
    'CNc1nn(C2CCN(CC3CCN(c4cc(C(C(=O)N5C[C@H](O)C[C@H]5C(=O)N[C@@H](C)c5ccc(-c6scnc6C)cc5)C(C)C)on4)CC3)CC2)c2cc(-c3ccccc3O)nnc12',
    'C#Cc1cccc2cc(O)cc(-c3ncc4c(N5CC6CCC(C5)N6)nc(OCC5(CN6CCN(CC7CCN(c8ccc9c(c8)C(=O)N(C8CCC(=O)NC8=O)C9=O)CC7)CC6)CC5)nc4c3F)c12',
    'CS(=O)(=O)N1CCc2cccc(Nc3nc(Nc4ccc(N5CCC(N6CCN(CC7CN(c8ccc9c(c8)C(=O)N(C8CCC(=O)NC8=O)C9=O)C7)CC6)CC5)c(F)c4)nc4[nH]ccc34)c21',
    'CC1(C)CCC(c2ccc(Cl)cc2)=C(CN2CCN3c4ccc(C(=O)NS(=O)(=O)c5ccc(N[C@H](CCN6CCN(C(=O)CCCCCC(=O)N[C@H](C(=O)N7C[C@H](O)C[C@H]7C(=O)NCc7ccc8c(c7)OCc7ncsc7-8)C(C)(C)C)CC6)CSc6ccccc6)c(S(=O)(=O)C(F)(F)F)c5)cc4OC[C@@H]3C2)C1',
    'CC1(C)CC(=C(c2ccc(N3CCN(CC4CN(c5ccc6c(c5)C(=O)N(C5CCC(=O)NC5=O)C6=O)C4)CC3)cc2)c2ccc3[nH]nc(F)c3c2)CC(C)(C)O1',
    'CC1(C)CC(=C(c2ccc(N3CCN(C(=O)C4CCN(c5ccc6c(c5)CN(C5CCC(=O)NC5=O)C6=O)CC4)CC3)nc2)c2ccc3[nH]nc(F)c3c2)CC(C)(C)O1',
    'Cc1c(Nc2nc3ccccc3s2)nnc2c1CCCN2c1ccc(-c2cnn(CC34CC5(C)CC(C)(C3)CC(OCCN(C)CCCCCCCCCCOc3ccc6c(c3)C(=O)N(C3CCC(=O)NC3=O)C6=O)(C5)C4)c2C)c(C(=O)O)n1',
    'Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)CCCCNC(=O)c2nc(N3CCCc4c3nnc(Nc3nc5ccccc5s3)c4C)ccc2-c2cnn(CC34CC5(C)CC(C)(C3)CC(OCCN3CCCC3)(C5)C4)c2C)C(C)(C)C)cc1',
    'CN1CCN([C@@H]2CCCN(c3c(F)cc(C(N)=O)c4[nH]c(-c5ccc(C6CCN(CC7CCN(c8ccc(C(=O)N[C@H]9CCC(=O)NC9=O)nc8)CC7)CC6)cc5)cc34)C2)C1=O',
    'CCOc1cc(N2CCC(N3CCN(CCc4cc(F)c([C@H]5CCC(=O)NC5=O)c(F)c4)CC3)CC2)c(CC)cc1Nc1ncc(Br)c(Nc2ccc(CC)c3c2N(S(C)(=O)=O)CC3)n1',
    'Cc1cc(-c2ncnc3[nH]c(-c4ccc(C5CCN(CC6CCN(c7ccc(N8CCC(=O)NC8=O)cc7F)CC6)CC5)cc4)cc23)ccc1CNC(=O)c1nc(C(C)(C)C)no1',
    'Cc1cc(N2CCC(CN3CCN(c4ccc(-c5cc6c(-c7ccc([C@@H](C)NC(=O)c8noc(C(C)(C)C)n8)c(C)c7F)ncnc6[nH]5)nc4)CC3)CC2)ccc1N1CCC(=O)NC1=O',
]

samples = [
    SampleInput(
        smiles=smiles,
        poi_name="AR",
        poi_sequence=AR_SEQ,
        ligase_name="CRBN",
        ligase_sequence=E3_SEQ,
        cell_line="Unknown",
        treatment_time=24.0,
        assay_type='Unknown',
    )
    for smiles in smiles_list
]

batch_results = predictor.predict_batch(samples, tasks=['bin'], verbose=True)

for i, res in enumerate(batch_results):
    print(f"Sample {i+1}:")
    print(f"  Ensemble bin: {res['bin'].summary()}")
    print('')
    if i >= 5:  # only print first 5 samples for brevity
        break

batch_results = predictor.predict_batch(samples, verbose=True)

for i, res in enumerate(batch_results):
    print(f"Sample {i+1}:")
    print(f"  Ensemble bin: {res['bin'].summary()}")
    print('')
    if i >= 5:  # only print first 5 samples for brevity
        break

The `predict()` method also accepts a simple dict with the same keys as `SampleInput` for convenience.

In [ ]:
sample = {
    'smiles': "CC1(C)[C@H](NC(=O)c2ccc(N3CCN(CCCOc4ccc(C(=O)NC5CCC(=O)NC5=O)nc4)CC3)nc2)C(C)(C)[C@H]1Oc1ccc(C#N)c(Cl)c1",
    'poi_name': "AR",
    'poi_sequence': AR_SEQ,
    'ligase_name': "CRBN",
    'ligase_sequence': E3_SEQ,
    'cell_line_id': "Unknown",
    'assay_time': 24.0,
    'assay': 'Unknown',
}

# predict() returns a dict: task_name → EnsemblePrediction
# EXAMPLE: Use task_results['dmax'] for Dmax predictions
task_results = predictor.predict(sample)

# For a single-task ensemble, extract the first (only) result
task_name = list(task_results.keys())[0]
result = task_results[task_name]
print(f"Task: {task_name}")
print(result.summary())

### 2.2 Accessing prediction details

The `EnsemblePrediction` object extracted from the task dict contains rich information.

Two kinds of 95% confidence interval are available:
- **Percentile CI**: non-parametric, shows "most models predict within this range"
- **SEM CI**: parametric, shows "the true average is likely in this range" (shrinks with more models)

In [ ]:
print(f"Weighted mean:             {result.weighted_mean[0]:.2f}")
print(f"Uncertainty (std):         {result.uncertainty_std[0]:.2f}")
print(f"95% CI (percentile):       [{result.ci_percentile_lower_95[0]:.2f}, {result.ci_percentile_upper_95[0]:.2f}]")
print(f"95% CI (SEM):              [{result.ci_sem_lower_95[0]:.2f}, {result.ci_sem_upper_95[0]:.2f}]")
print(f"Prediction variance:       {result.prediction_variance[0]:.4f}")
print(f"Prediction range:          {result.prediction_range[0]:.2f}")
print(f"IQR:                       {result.prediction_iqr[0]:.2f}")
print(f"Models contributing:       {len(result.model_names)}")

In [ ]:
# Show individual model predictions
individual_df = pd.DataFrame([
    {
        "model": name,
        "prediction": pred[0],
        "weight": result.weights.get(name, 0),
    }
    for name, pred in result.individual_predictions.items()
]).sort_values("weight", ascending=False).reset_index(drop=True)

individual_df

## 3. Batch Predictions

### 3.1 Predicting from a list of `SampleInput`s

In [ ]:
SMARCA2_SEQ = "MSTPTDPGAMPHPGPSPGPGPSPGPILGPSPGPGPSPGSVHSMMGPSPGPPSVSHPMPTMGSTDFPQEGMHQMHKPIDGIHDKGIVEDIHCGSMKGTGMRPPHPGMGPPQSPMDQHSQGYMSPHPSPLGAPEHVSSPMSGGGPTPPQMPPSQPGALIPGDPQAMSQPNRGPSPFSPVQLHQLRAQILAYKMLARGQPLPETLQLAVQGKRTLPGLQQQQQQQQQQQQQQQQQQQQQQQPQQQPPQPQTQQQQQPALVNYNRPSGPGPELSGPSTPQKLPVPAPGGRPSPAPPAAAQPPAAAVPGPSVPQPAPGQPSPVLQLQQKQSRISPIQKPQGLDPVEILQEREYRLQARIAHRIQELENLPGSLPPDLRTKATVELKALRLLNFQRQLRQEVVACMRRDTTLETALNSKAYKRSKRQTLREARMTEKLEKQQKIEQERKRRQKHQEYLNSILQHAKDFKEYHRSVAGKIQKLSKAVATWHANTEREQKKETERIEKERMRRLMAEDEEGYRKLIDQKKDRRLAYLLQQTDEYVANLTNLVWEHKQAQAAKEKKKRRRRKKKAEENAEGGESALGPDGEPIDESSQMSDLPVKVTHTETGKVLFGPEAPKASQLDAWLEMNPGYEVAPRSDSEESDSDYEEEDEEEESSRQETEEKILLDPNSEEVSEKDAKQIIETAKQDVDDEYSMQYSARGSQSYYTVAHAISERVEKQSALLINGTLKHYQLQGLEWMVSLYNNNLNGILADEMGLGKTIQTIALITYLMEHKRLNGPYLIIVPLSTLSNWTYEFDKWAPSVVKISYKGTPAMRRSLVPQLRSGKFNVLLTTYEYIIKDKHILAKIRWKYMIVDEGHRMKNHHCKLTQVLNTHYVAPRRILLTGTPLQNKLPELWALLNFLLPTIFKSCSTFEQWFNAPFAMTGERVDLNEEETILIIRRLHKVLRPFLLRRLKKEVESQLPEKVEYVIKCDMSALQKILYRHMQAKGILLTDGSEKDKKGKGGAKTLMNTIMQLRKICNHPYMFQHIEESFAEHLGYSNGVINGAELYRASGKFELLDRILPKLRATNHRVLLFCQMTSLMTIMEDYFAFRNFLYLRLDGTTKSEDRAALLKKFNEPGSQYFIFLLSTRAGGLGLNLQAADTVVIFDSDWNPHQDLQAQDRAHRIGQQNEVRVLRLCTVNSVEEKILAAAKYKLNVDQKVIQAGMFDQKSSSHERRAFLQAILEHEEENEEEDEVPDDETLNQMIARREEEFDLFMRMDMDRRREDARNPKRKPRLMEEDELPSWIIKDDAEVERLTCEEEEEKIFGRGSRQRRDVDYSDALTEKQWLRAIEDGNLEEMEEEVRLKKRKRRRNVDKDPAKEDVEKAKKRRGRPPAEKLSPNPPKLTKQMNAIIDTVINYKDRCNVEKVPSNSQLEIEGNSSGRQLSEVFIQLPSRKELPEYYELIRKPVDFKKIKERIRNHKYRSLGDLEKDVMLLCHNAQTFNLEGSQIYEDSIVLQSVFKSARQKIAKEEESEDESNEEEEEEDEEESESEAKSVKVKIKLNKKDDKGRDKGKGKKRPNRGKAKPVVSDFDSDEEQDEREQSEGSGTDDE"
BTK_SEQ = "MAAVILESIFLKRSQQKKKTSPLNFKKRLFLLTVHKLSYYEYDFERGRRGSKKGSIDVEKITCVETVVPEKNPPPERQIPRRGEESSEMEQISIIERFPYPFQVVYDEGPLYVFSPTEELRKRWIHQLKNVIRYNSDLVQKYHPCFWIDGQYLCCSQTAKNAMGCQILENRNGSLKPGSSHRKTKKPLPPTPEEDQILKKPLPPEPAAAPVSTSELKKVVALYDYMPMNANDLQLRKGDEYFILEESNLPWWRARDKNGQEGYIPSNYVTEAEDSIEMYEWYSKHMTRSQAEQLLKQEGKEGGFIVRDSSKAGKYTVSVFAKSTGDPQGVIRHYVVCSTPQSQYYLAEKHLFSTIPELINYHQHNSAGLISRLKYPVSQQNKNAPSTAGLGYGSWEIDPKDLTFLKELGTGQFGVVKYGKWRGQYDVAIKMIKEGSMSEDEFIEEAKVMMNLSHEKLVQLYGVCTKQRPIFIITEYMANGCLLNYLREMRHRFQTQQLLEMCKDVCEAMEYLESKQFLHRDLAARNCLVNDQGVVKVSDFGLSRYVLDDEYTSSVGSKFPVRWSPPEVLMYSKFSSKSDIWAFGVLMWEIYSLGKMPYERFTNSETAEHIAQGLRLYRPHLASEKVYTIMYSCWHEKADERPTFKILLSNILDVMDEES"

samples = [
    SampleInput(
        smiles="COc1ccc(C2=NN(c3ccc(C(=O)Nc4cccc(-c5nc6ccccn6c5C)c4)cc3)C(c3ccc(F)cc3)C2)cc1",
        poi_sequence=AR_SEQ, ligase_name="CRBN", ligase_sequence=E3_SEQ,
        treatment_time=24.0, assay_type='Unknown', cell_line='Unknown',
    ),
    SampleInput(
        smiles="CC(C)(C)OC(=O)N1CCC(NCc2cccc(-c3ccc4[nH]c(N)nc4c3)c2)CC1",
        poi_sequence=BTK_SEQ, ligase_name="CRBN", ligase_sequence=E3_SEQ,
        treatment_time=6.0, assay_type='Unknown', cell_line='Unknown',
    ),
    SampleInput(
        smiles="Cc1ncnc2c1cnn2-c1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1",
        poi_sequence=SMARCA2_SEQ, ligase_name="CRBN", ligase_sequence=E3_SEQ,
        treatment_time=18.0, assay_type='Unknown', cell_line='Unknown',
    ),
]

# predict_batch returns List[Dict[str, EnsemblePrediction]]
batch_results = predictor.predict_batch(samples)

for i, task_dict in enumerate(batch_results):
    if task_dict is not None:
        for t, r in task_dict.items():
            print(f"Sample {i} [{t}]: {r.weighted_mean[0]:.2f} ± {r.uncertainty_std[0]:.2f}")
    else:
        print(f"Sample {i}: prediction failed")

## 4. Visualising Predictions

Below we show how to plot the individual model predictions and the
ensemble uncertainty.

In [ ]:
# Re-predict a sample and extract the first task's result
task_results_viz = predictor.predict(samples[0])
result = next(iter(task_results_viz.values()))

# Collect per-model predictions
model_names = list(result.individual_predictions.keys())
preds = np.array([result.individual_predictions[n][0] for n in model_names])
weights = np.array([result.weights[n] for n in model_names])

# Sort by prediction value
order = np.argsort(preds)
model_names = [model_names[i] for i in order]
preds = preds[order]
weights = weights[order]

# Shorten long model names for display
short_names = [n.replace("model=", "")[:60] for n in model_names]

fig, ax = plt.subplots(figsize=(10, max(4, len(model_names) * 0.35)))
colors = plt.cm.Blues(weights / weights.max() * 0.7 + 0.3)
ax.barh(range(len(preds)), preds, color=colors, edgecolor="navy", alpha=0.8)
ax.axvline(result.weighted_mean[0], color="red", ls="--", lw=2,
           label=f"Ensemble mean: {result.weighted_mean[0]:.1f}")
ax.axvspan(result.ci_percentile_lower_95[0], result.ci_percentile_upper_95[0],
           alpha=0.12, color="red", label="95% CI (percentile)")
ax.axvspan(result.ci_sem_lower_95[0], result.ci_sem_upper_95[0],
           alpha=0.15, color="green", label="95% CI (SEM)")
ax.set_yticks(range(len(short_names)))
ax.set_yticklabels(short_names, fontsize=7)
ax.set_xlabel("Prediction")
ax.set_title("Individual model predictions")
ax.legend(loc="lower right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Serialisation & Export

Prediction results can be serialised to a dictionary (for JSON export) or
inspected via the `.summary()` convenience method.

In [ ]:
result_dict = result.to_dict()

# Pretty-print a subset
subset = {k: result_dict[k] for k in [
    "weighted_mean",
    "ci_percentile_lower_95", "ci_percentile_upper_95",
    "ci_sem_lower_95", "ci_sem_upper_95",
    "n_models", "task", "label_name",
]}
print(json.dumps(subset, indent=2))

## 6. Required Inputs per Model

Different models may require different input features (e.g., some need
POI sequence, some need cell descriptions). You can inspect what each
model needs.

In [ ]:
required, _ = predictor.get_required_inputs()

print(required)

## 7. Updating Weights

You can update the ensemble weights at any time without reloading the
models.  Weights are automatically re-normalised to sum to 1.

In [ ]:
# Give equal weight to all models
equal_weights = {name: 1.0 for name in predictor.models}
predictor.update_weights(equal_weights)

res_eq = next(iter(predictor.predict(samples[0]).values()))
print(f"Equal weights -> {res_eq.weighted_mean[0]:.2f} ± {res_eq.uncertainty_std[0]:.2f}")

# Give all weight to XGBoost models only
xgb_only = {
    name: 1.0 if predictor.model_types[name] == "xgboost" else 0.0
    for name in predictor.models
}
# Filter out zero-weight entries before updating
xgb_only = {k: v for k, v in xgb_only.items() if v > 0}
predictor.update_weights(xgb_only)

res_xgb = next(iter(predictor.predict(samples[0]).values()))
print(f"XGBoost only  -> {res_xgb.weighted_mean[0]:.2f} ± {res_xgb.uncertainty_std[0]:.2f}")

# Restore equal weights
predictor.update_weights(equal_weights)